In [1]:
%pprint
%load_ext supriya.ext.ipython

import warnings

warnings.filterwarnings(action="default", module="supriya")

from supriya.scsynth import kill

kill()

Pretty printing has been turned OFF


In [2]:
# soundcheck

March 18th, 2025

# Supriya: a Python API for SuperCollider

Joséphine Wolf Oberholtzer (she / her)

## Introduction

- who am i
- what is supriya
- what isn't it
- what is this presentation
- what isn't this presentation
- what's in a name
- presentation overview

## Servers

### Import supriya

In [3]:
import supriya

In [4]:
dir(supriya)

['AddAction', 'AsyncClock', 'AsyncOfflineClock', 'AsyncServer', 'BaseClock', 'BaseServer', 'Buffer', 'BufferGroup', 'Bus', 'BusGroup', 'CalculationRate', 'Clock', 'ClockContext', 'Context', 'DoneAction', 'Envelope', 'Group', 'HeaderFormat', 'Node', 'OfflineClock', 'Options', 'OscBundle', 'OscCallback', 'OscMessage', 'ParameterRate', 'Pattern', 'SampleFormat', 'Score', 'Server', 'ServerLifecycleEvent', 'ServerShutdownEvent', 'Synth', 'SynthDef', 'SynthDefBuilder', 'TimeUnit', 'UGen', 'UGenOperable', 'UGenVector', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', '__version_info__', '_version', 'assets', 'assets_path', 'clocks', 'contexts', 'conversions', 'default', 'enums', 'exceptions', 'ext', 'graph', 'io', 'osc', 'output_path', 'patterns', 'play', 'plot', 'render', 'sclang', 'scsynth', 'synthdef', 'typing', 'ugens', 'utils']

### Import `Server`

In [5]:
from supriya import Server

In [6]:
server = Server()

In [7]:
dir(server)

['__abstractmethods__', '__annotations__', '__class__', '__contains__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__firstlineno__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__static_attributes__', '__str__', '__subclasshook__', '__weakref__', '_abc_impl', '_add_node_to_children', '_add_request_with_completion', '_add_requests', '_allocate_id', '_apply_completions', '_audio_bus_allocator', '_boot_future', '_boot_status', '_buffer_allocator', '_buffers', '_client_id', '_contexts', '_control_bus_allocator', '_exit_future', '_free_id', '_get_allocator', '_get_moment', '_get_next_sync_id', '_get_request_context', '_handle_done_b_alloc', '_handle_done_b_alloc_read', '_handle_done_b_alloc_read_channel', '_handle_done_b_free', '_handle_fail', '_handle_n_end', '_

### Server options

In [8]:
server

<Server OFFLINE [/Applications/SuperCollider.app/Contents/Resources/scsynth -R 0 -l 1 -u 57110]>

In [9]:
# TODO: Make Options a dataclass
server.options

Options(audio_bus_channel_count=1024, block_size=64, buffer_count=1024, control_bus_channel_count=16384, executable=None, hardware_buffer_size=None, initial_node_id=1000, input_bus_channel_count=8, input_device=None, input_stream_mask='', ip_address='127.0.0.1', load_synthdefs=True, maximum_logins=1, maximum_node_count=1024, maximum_synthdef_count=1024, memory_locking=False, memory_size=8192, output_bus_channel_count=8, output_device=None, output_stream_mask='', password=None, port=57110, protocol='udp', random_number_generator_count=64, realtime=True, remote_control_volume=False, restricted_path=None, safety_clip=None, sample_rate=None, threads=None, ugen_plugins_path=None, verbosity=0, wire_buffer_count=64, zero_configuration=False)

### Boot the server

In [10]:
server.boot()

<Server ONLINE [/Applications/SuperCollider.app/Contents/Resources/scsynth -R 0 -l 1 -u 57110]>

### Inspect the server

In [11]:
server.status

StatusInfo(actual_sample_rate=44100.04827922403, average_cpu_usage=0.033147744834423065, group_count=2, peak_cpu_usage=0.06391054391860962, synth_count=0, synthdef_count=32, target_sample_rate=44100.0, ugen_count=0)

In [12]:
print(tree := server.query_tree())

NODE TREE 0 group
    1 group


In [13]:
# this is actually a query tree object, not just a string
# ... which is helpful in more complex unit testing situations
# ... because the tree can be annotated with information beyond what scsynth provides
tree

QueryTreeGroup(node_id=0, annotation=None, children=[QueryTreeGroup(node_id=1, annotation=None, children=[])])

### Reset the server

In [14]:
# without quitting...
# - delete all nodes and synthdefs
# - and clear all allocators
server.reset()

<Server ONLINE [/Applications/SuperCollider.app/Contents/Resources/scsynth -R 0 -l 1 -u 57110]>

In [16]:
print(server.query_tree())

NODE TREE 0 group
    1 group


### Quit the server

In [27]:
server.quit()

<Server OFFLINE [/Applications/SuperCollider.app/Contents/Resources/scsynth -R 0 -l 2 -u 57110]>

### Boot with options

In [28]:
server.boot(maximum_logins=2)

/Users/josephine/Source/github.com/supriya-project/supriya/supriya/osc/threaded.py:187: ResourceWarning: unclosed <socket.socket fd=71, family=2, type=2, proto=0, laddr=('0.0.0.0', 64902)>
  self.osc_server = self._server_factory(ip_address, port)


<Server ONLINE [/Applications/SuperCollider.app/Contents/Resources/scsynth -R 0 -l 2 -u 57110]>

### Multiple users

In [29]:
# We can create a handle to a second server proxy,
# pointed back at the same IP address and port as the first
other_server = Server()
other_server

<Server OFFLINE [/Applications/SuperCollider.app/Contents/Resources/scsynth -R 0 -l 1 -u 57110]>

In [30]:
server  # note the port is the same as `other_server`

<Server ONLINE [/Applications/SuperCollider.app/Contents/Resources/scsynth -R 0 -l 2 -u 57110]>

In [31]:
other_server.connect()  # connect to the original server via .connect()

<Server ONLINE [/Applications/SuperCollider.app/Contents/Resources/scsynth -R 0 -l 1 -u 57110]>

In [33]:
print(f"{server.client_id=}")
print(f"{other_server.client_id=}")

server.client_id=0
other_server.client_id=1


In [26]:
other_server.disconnect()  # disconnect from the original server

ServerOffline: 

In [ ]:
server  # the original server remains online

n.b. it's not clear to me how to release a client ID once consumed

### Lifecycle events

In [ ]:
def on_event(event):
    print(event)

for event_type in supriya.ServerLifecycleEvent:
    server.on(event_type, on_event)
    other_server.on(event_type, on_event)

In [ ]:
server.reboot()

In [ ]:
# this will panic
other_server.boot()

In [ ]:
server.quit()

## Entities

### Groups

In [ ]:
server = Server().boot()

In [ ]:
# add a group
group = server.add_group()
group

In [ ]:
print(dir(group))

In [ ]:
print(server.query_tree())

In [ ]:
# add a group to the group
child_group = group.add_group()
print(server.query_tree())

In [ ]:
# move the child group into the default group
child_group.move(target_node=server.default_group, add_action="ADD_TO_TAIL")
print(server.query_tree())

In [ ]:
# free the original parent group
group.free()
print(server.query_tree())

### Synths

In [ ]:
# add a synth (this will fail, silently)
synth = child_group.add_synth(synthdef=supriya.default)

In [ ]:
# we have a proxy to a synth
synth

In [ ]:
# but nothing on the server, because the request failed
print(server.query_tree())

In [ ]:
# using on_completion keyword argument
server.add_synthdefs(
    supriya.default,
    on_completion=lambda server: child_group.add_synth(synthdef=supriya.default),
) 
server.sync()
print(server.query_tree())

In [ ]:
# using completion context manager
with server.at():
    with server.add_synthdefs(supriya.default):
        synth = child_group.add_synth(synthdef=supriya.default, frequency=666)
server.sync()
print(server.query_tree())

In [ ]:
synth.set(frequency=555)

In [ ]:
synth.free()

In [ ]:
child_group.set(gate=0)

### Buses

- calculation rates
- allocate and free
- buses vs bus groups
- shared memory

In [36]:
# add a control bus
control_bus = server.add_bus()
control_bus

Bus(context=<Server ONLINE [/Applications/SuperCollider.app/Contents/Resources/scsynth -R 0 -l 2 -u 57110]>, id_=0, calculation_rate=CalculationRate.CONTROL)

In [37]:
# add an audio bus
audio_bus = server.add_bus("audio")
audio_bus

Bus(context=<Server ONLINE [/Applications/SuperCollider.app/Contents/Resources/scsynth -R 0 -l 2 -u 57110]>, id_=16, calculation_rate=CalculationRate.AUDIO)

In [38]:
# add a bus group
control_bus_group = server.add_bus_group(count=4)
control_bus_group

BusGroup(context=<Server ONLINE [/Applications/SuperCollider.app/Contents/Resources/scsynth -R 0 -l 2 -u 57110]>, id_=1, calculation_rate=CalculationRate.CONTROL, count=4)

In [39]:
# a bus group aggregates together bus objects
for bus in control_bus_group:
    print(bus)

Bus(context=<Server ONLINE [/Applications/SuperCollider.app/Contents/Resources/scsynth -R 0 -l 2 -u 57110]>, id_=1, calculation_rate=CalculationRate.CONTROL)
Bus(context=<Server ONLINE [/Applications/SuperCollider.app/Contents/Resources/scsynth -R 0 -l 2 -u 57110]>, id_=2, calculation_rate=CalculationRate.CONTROL)
Bus(context=<Server ONLINE [/Applications/SuperCollider.app/Contents/Resources/scsynth -R 0 -l 2 -u 57110]>, id_=3, calculation_rate=CalculationRate.CONTROL)
Bus(context=<Server ONLINE [/Applications/SuperCollider.app/Contents/Resources/scsynth -R 0 -l 2 -u 57110]>, id_=4, calculation_rate=CalculationRate.CONTROL)


In [40]:
# buses and bus groups all can be expressed as map symbols
for bus_like in (control_bus, audio_bus, control_bus_group):
    print(bus_like.map_symbol())

c0
a16
c1


In [41]:
# we can map buses to synth controls
synth.map(amplitude=control_bus, frequency=audio_bus)
print(server.query_tree())

NameError: name 'synth' is not defined

In [44]:
# we can release the bus IDs back to the allocator pools
for bus_like in (control_bus, audio_bus, control_bus_group):
    bus_like.free()

### Buffers

- allocate and free
- read sound files
- generating
- plotting

### Enums

In [43]:
# calculation rates are expressed via an enum
from supriya import CalculationRate

for rate in sorted(CalculationRate):
    print(repr(rate), rate)

CalculationRate.SCALAR 0
CalculationRate.CONTROL 1
CalculationRate.AUDIO 2
CalculationRate.DEMAND 3


## Messages

In [ ]:
server.reset()

In [ ]:
server.osc_protocol

In [ ]:
with server.osc_protocol.capture() as transcript:
    with server.at():
        with server.add_synthdefs(supriya.default):
            group = server.add_group()
            synth = group.add_synth(synthdef=supriya.default, frequency=666)

In [ ]:
for x in transcript: print(x)

### OSC messages & bundles

### Sending messages

In [ ]:
server.send([])

### OSC protocols

In [ ]:
server.osc_protocol

### Capturing IO

In [ ]:
with server.osc_protocol.capture() as transcript:
    with server.at():
        group = server.add_group()
        with server.add_synthdefs(supriya.default):
            synth = group.add_synth(synthdef=supriya.default, frequency=666)
    server.sync()

In [ ]:
for entry in transcript:
    print(entry)

### Moments & completions

### Requests & responses

## Synthdefs

### Building SynthDefs (1)

In [ ]:
r"""
SynthDef(\simple, { arg out=0, freq=440;
	Out.ar(out, SinOsc.ar(freq));
});
"""

In [ ]:
# A simple SynthDef using the builder pattern
from supriya.ugens import SynthDefBuilder
from supriya.ugens import Out, SinOsc

with SynthDefBuilder(freq=440, out=0) as builder:
    source = SinOsc.ar(frequency=builder["freq"])
    Out.ar(bus=builder["out"], source=[source, source])

simple_synthdef = builder.build(name="simple")
simple_synthdef

### Building SynthDefs (2)

In [ ]:
r"""
*makeDefaultSynthDef {
    SynthDef(\default, { arg out=0, freq=440, amp=0.1, pan=0, gate=1;
        var z;
        z = LPF.ar(
            Mix.new(VarSaw.ar(freq + [0, Rand(-0.4,0.0), Rand(0.0,0.4)], 0, 0.3, 0.3)),
            XLine.kr(Rand(4000,5000), Rand(2500,3200), 1)
        ) * Linen.kr(gate, 0.01, 0.7, 0.3, 2);
        OffsetOut.ar(out, Pan2.ar(z, pan, amp));
    }, [\ir]).add;
}
"""

In [ ]:
# More imports
from supriya.enums import DoneAction, ParameterRate
from supriya.ugens import (
    LPF,
    Linen,
    Mix,
    OffsetOut,
    Pan2,
    Parameter,
    Rand,
    SynthDefBuilder,
    VarSaw,
    XLine,
)

In [ ]:
# Define a builder
builder = SynthDefBuilder(
    out=Parameter(rate=ParameterRate.SCALAR, value=0),
    amplitude=0.1,
    frequency=440,
    gate=1,
    pan=0.5,
)

In [ ]:
# Use the builder as a context manager
with builder:
    linen = Linen.kr(
        attack_time=0.01,
        done_action=DoneAction.FREE_SYNTH,
        gate=builder["gate"],
        release_time=0.3,
        sustain_level=0.7,
    )

In [ ]:
# Use the builder again
with builder:
    low_pass = LPF.ar(
        source=Mix.new(
            VarSaw.ar(
                frequency=builder["frequency"]
                + (
                    0,
                    Rand.ir(minimum=-0.4, maximum=0.0),
                    Rand.ir(minimum=0.0, maximum=0.4),
                ),
                width=0.3,
            )
        )
        * 0.3,
        frequency=XLine.kr(
            start=Rand.ir(minimum=4000, maximum=5000),
            stop=Rand.ir(minimum=2500, maximum=3200),
        ),
    )

In [ ]:
# And again and again
with builder:
    panner = Pan2.ar(
        source=low_pass * linen * builder["amplitude"], position=builder["pan"]
    )

with builder:
    OffsetOut.ar(bus=builder["out"], source=panner)

In [ ]:
default = builder.build(name="default")
default

### The `synthdef` decorator

In [ ]:
# N.B. I'm not fond of this one because of
# a) how magical it is (not very, but just enough) but mainly because
# b) it makes type-checking difficult
from supriya.ugens import synthdef

In [ ]:
@synthdef("ir")
def default_decorated(out=0, amplitude=0.1, frequency=440, gate=1, pan=0.5):
    linen = Linen.kr(
        attack_time=0.01,
        done_action=DoneAction.FREE_SYNTH,
        gate=gate,
        release_time=0.3,
        sustain_level=0.7,
    )
    low_pass = LPF.ar(
        source=Mix.new(
            VarSaw.ar(
                frequency=frequency
                + (
                    0,
                    Rand.ir(minimum=-0.4, maximum=0.0),
                    Rand.ir(minimum=0.0, maximum=0.4),
                ),
                width=0.3,
            )
        )
        * 0.3,
        frequency=XLine.kr(
            start=Rand.ir(minimum=4000, maximum=5000),
            stop=Rand.ir(minimum=2500, maximum=3200),
        ),
    )
    panner = Pan2.ar(
        source=low_pass * linen * amplitude, position=pan
    )
    _ = OffsetOut.ar(bus=out, source=panner)

In [ ]:
default_decorated

In [ ]:
from supriya.ugens import Out, SinOsc

@synthdef()
def foo(out=0):
    print(repr(out))
    _ = Out.ar(source=SinOsc.kr())

foo

### Graphing SynthDefs

In [ ]:
print(default)

In [ ]:
from supriya import graph

_ = graph(default)

### SynthDef internals

In [ ]:
dir(default)

In [ ]:
default.name

In [ ]:
default.anonymous_name

In [ ]:
default.effective_name

In [ ]:
default.parameters

In [ ]:
default.ugens

In [ ]:
default.has_gate

### UGen methods

In [ ]:
dir(SinOsc)

### SynthDef (de)compilation

In [ ]:
# SynthDefs compile to byte strings
compiled = default.compile()
compiled

In [ ]:
# valid byte strings can be decompiled back into SynthDefs
from supriya.ugens import decompile_synthdef

decompiled = decompile_synthdef(compiled)
decompiled

In [ ]:
# sanity-check: the decompiled SynthDef is not the same in memory
default is decompiled

### Compiling via sclang

In [ ]:
# Supriya provides utilities for compiling via sclang.
# This is intended for validating its own logic vs sclang (as a reference spec).
from supriya.ugens import SuperColliderSynthDef

sc_synthdef = SuperColliderSynthDef(
    "foo", "Out.ar(0, SinOsc.ar(freq: 420) * SinOsc.ar(freq: 440))"
)
sc_compiled_synthdef = sc_synthdef.compile()  # return bytes
sc_compiled_synthdef

In [ ]:
# The sclang-derived SynthDef byte string can be decompiled back into a SynthDef.
print(decompile_synthdef(sc_compiled_synthdef))

### UGen metaprogramming

In [ ]:
from supriya.ugens import UGen, param, ugen

# A dupe of SinOsc
@ugen(ar=True, kr=True, is_pure=True)
class AnotherSinOsc(UGen):
    frequency = param(440.0)
    phase = param(0.0)

In [ ]:
AnotherSinOsc.ar()

In [ ]:
AnotherSinOsc.kr()

In [ ]:
# This won't work because ir=True wasn't set
AnotherSinOsc.ir()

In [ ]:
# A dupe of Out
@ugen(ar=True, kr=True, is_output=True, channel_count=0, fixed_channel_count=True)
class AnotherOut(UGen):
    bus = param(0)
    source = param(unexpanded=True)

AnotherOut.ar(source=AnotherSinOsc.ar())

In [ ]:
from supriya.ugens.pv import PV_ChainUGen

# A dupe of PV_BinShift
@ugen(kr=True, is_width_first=True)
class AnotherPV_BinShift(PV_ChainUGen):
    pv_chain = param()
    stretch = param(1.0)
    shift = param(0.0)
    interpolate = param(0)

# This won't work because of missing pv_chain argument
AnotherPV_BinShift.kr()

In [ ]:
"""
def ugen(
    *,
    ar: bool = False,
    kr: bool = False,
    ir: bool = False,
    dr: bool = False,
    new: bool = False,
    has_done_flag: bool = False,
    is_input: bool = False,
    is_multichannel: bool = False,
    is_output: bool = False,
    is_pure: bool = False,
    is_width_first: bool = False,
    channel_count: int = 1,
    fixed_channel_count: bool = False,
    signal_range: Optional[int] = None,
) -> Callable[[Type["UGen"]], Type["UGen"]]:
    ...
"""

## Clocks & Patterns

## Non-Realtime

- server
- session
- mutations vs queries
- rendering (async)

## Concurrency

- server
- async server
- threaded concurrency
- event loop / coroutine concurrency

## Mixers

## Cleanliness

- docs
- typing
- testing
- ci/cd